In [5]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing import image_dataset_from_directory
from collections import Counter
import pandas as pd

In [6]:
IMAGE_SIZE = (224, 224)   # Resize all images to this size
BATCH_SIZE = 32
DATA_DIR = "images"       # Your main folder

In [9]:
train_ds = image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.3,
    subset="training",
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)

Found 1373 files belonging to 6 classes.
Using 962 files for training.


In [10]:
val_ds = image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.3,
    subset="validation",
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)

Found 1373 files belonging to 6 classes.
Using 411 files for validation.


In [11]:
val_batches = tf.data.experimental.cardinality(val_ds).numpy()
test_ds = val_ds.skip(val_batches // 2)
val_ds = val_ds.take(val_batches // 2)

In [12]:
val_labels = []
for images, labels in val_ds:
  val_labels.extend(labels.numpy())

val_labels_counts = Counter(val_labels)

test_labels = []
for images, labels in test_ds:
  test_labels.extend(labels.numpy())

test_labels_counts = Counter(test_labels)

print('Validation Data:', val_labels_counts, '-> size =', sum(val_labels_counts.values()))
print('Testing Data:', test_labels_counts, '-> size =', sum(test_labels_counts.values()))

Validation Data: Counter({np.int32(4): 44, np.int32(3): 41, np.int32(0): 38, np.int32(2): 35, np.int32(5): 34}) -> size = 192
Testing Data: Counter({np.int32(5): 56, np.int32(4): 48, np.int32(0): 45, np.int32(3): 38, np.int32(2): 32}) -> size = 219


2025-06-07 21:41:06.275310: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-06-07 21:41:06.349218: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [13]:
num_classes = len(train_ds.class_names)
assert(num_classes  == len(val_labels_counts) == len(test_labels_counts))

AssertionError: 

In [14]:
model = Sequential()
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(*IMAGE_SIZE, 3)))
# model.add(BatchNormalization())
model.add(MaxPooling2D())

model.add(Conv2D(64, (3, 3), activation='relu'))
# model.add(BatchNormalization())
model.add(MaxPooling2D())

model.add(Conv2D(128, (3, 3), activation='relu'))
# model.add(BatchNormalization())
model.add(MaxPooling2D())

model.add(Flatten())
model.add(Dense(128, activation='relu'))
# model.add(BatchNormalization())
model.add(Dropout(0.5))

model.add(Dense(num_classes, activation='softmax'))

/Users/amin/Projects/CSCI-184-Project/.venv/lib/python3.9/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [27]:
model.compile(optimizer=Adam(),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [28]:
model.fit(train_ds, validation_data=val_ds, epochs=5)

Epoch 1/5
31/31 ━━━━━━━━━━━━━━━━━━━━ 21s 586ms/step - accuracy: 0.2082 - loss: 242.9983 - val_accuracy: 0.2188 - val_loss: 1.6046
Epoch 2/5
31/31 ━━━━━━━━━━━━━━━━━━━━ 17s 561ms/step - accuracy: 0.3257 - loss: 1.5554 - val_accuracy: 0.2344 - val_loss: 1.5936
Epoch 3/5
31/31 ━━━━━━━━━━━━━━━━━━━━ 17s 563ms/step - accuracy: 0.4295 - loss: 1.4584 - val_accuracy: 0.2240 - val_loss: 1.6975
Epoch 4/5
31/31 ━━━━━━━━━━━━━━━━━━━━ 17s 555ms/step - accuracy: 0.4738 - loss: 1.3033 - val_accuracy: 0.2760 - val_loss: 1.7329
Epoch 5/5
31/31 ━━━━━━━━━━━━━━━━━━━━ 18s 571ms/step - accuracy: 0.5981 - loss: 1.0445 - val_accuracy: 0.2812 - val_loss: 1.7708


In [29]:
test_loss, test_acc = model.evaluate(test_ds, verbose=2)
print(f"Test accuracy: {test_acc:.4f}")

7/7 - 1s - 135ms/step - accuracy: 0.2785 - loss: 1.8065
Test accuracy: 0.2785


Grid

In [13]:
def get_data(split=0.3, seed=42, batch_size=32):
  train_ds = image_dataset_from_directory(
    DATA_DIR,
    validation_split=split,
    subset="training",
    seed=seed,
    image_size=IMAGE_SIZE,
    batch_size=batch_size
  )
  
  val_ds_temp = image_dataset_from_directory(
    DATA_DIR,
    validation_split=split,
    subset="validation",
    seed=seed,
    image_size=IMAGE_SIZE,
    batch_size=batch_size
  )

  val_batches = tf.data.experimental.cardinality(val_ds_temp).numpy()
  test_ds = val_ds_temp.skip(val_batches // 2)
  val_ds = val_ds_temp.take(val_batches // 2)

  val_labels = []
  for _, labels in val_ds:
    val_labels.extend(labels.numpy())
  val_labels_counts = Counter(val_labels)
  test_labels = []
  for _, labels in test_ds:
    test_labels.extend(labels.numpy())
  test_labels_counts = Counter(test_labels)
  print('Validation Data:', val_labels_counts, '-> size =', sum(val_labels_counts.values()))
  print('Testing Data:', test_labels_counts, '-> size =', sum(test_labels_counts.values()))

  num_classes = len(train_ds.class_names)
  assert(num_classes  == len(val_labels_counts) == len(test_labels_counts))
  
  return train_ds, val_ds, test_ds

def use_model_1(num_classes):
  model = Sequential()
  model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(*IMAGE_SIZE, 3)))
  model.add(MaxPooling2D())

  model.add(Conv2D(64, (3, 3), activation='relu'))
  model.add(MaxPooling2D())

  model.add(Conv2D(128, (3, 3), activation='relu'))
  model.add(MaxPooling2D())

  model.add(Flatten())
  model.add(Dense(128, activation='relu'))
  model.add(Dropout(0.5))

  model.add(Dense(num_classes, activation='softmax'))

  model.compile(optimizer=Adam(),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

  return model

def use_model_2(num_classes):
  model = Sequential()
  model.add(Conv2D(16, (3, 3), activation='relu', input_shape=(*IMAGE_SIZE, 3)))
  model.add(MaxPooling2D())

  model.add(Conv2D(32, (3, 3), activation='relu'))
  model.add(MaxPooling2D())

  model.add(Conv2D(64, (3, 3), activation='relu'))
  model.add(MaxPooling2D())

  model.add(Flatten())
  model.add(Dense(64, activation='relu'))
  model.add(Dropout(0.5))

  model.add(Dense(num_classes, activation='softmax'))

  model.compile(optimizer=Adam(),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

  return model

def use_model_3(num_classes):
  model = Sequential()
  model.add(Conv2D(16, (3, 3), activation='relu', input_shape=(*IMAGE_SIZE, 3)))
  model.add(MaxPooling2D())

  model.add(Conv2D(32, (3, 3), activation='relu'))
  model.add(MaxPooling2D())

  model.add(Flatten())
  model.add(Dense(64, activation='relu'))
  model.add(Dropout(0.5))

  model.add(Dense(num_classes, activation='softmax'))

  model.compile(optimizer=Adam(),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

  return model

In [16]:
results = []

model_functions = {
    'Model_1': use_model_1,
    'Model_2': use_model_2,
    'Model_3': use_model_3
}

seeds = [1, 2]
epochs_list = [2, 3, 4]
batch_sizes = [16, 32]

In [17]:
for model_name, model_func in model_functions.items():
  for seed in seeds:
    for batch_size in batch_sizes:
      train_ds, val_ds, test_ds = get_data(seed=seed, batch_size=batch_size)
      num_classes = len(train_ds.class_names)  # Needed inside each model function
      for epochs in epochs_list:
          print(f"\nTraining {model_name} | Seed: {seed} | Batch: {batch_size} | Epochs: {epochs}")
          model = model_func(num_classes)
          history = model.fit(train_ds, validation_data=val_ds, epochs=epochs, verbose=1)
          val_acc = history.history['val_accuracy'][-1]
          test_loss, test_acc = model.evaluate(test_ds, verbose=0)
          result = {
              'model': model_name,
              'seed': seed,
              'batch_size': batch_size,
              'epochs': epochs,
              'val_acc': val_acc,
              'test_acc': test_acc,
              'test_loss': test_loss
            }
          print(result)
          results.append(result)

Found 1372 files belonging to 5 classes.
Using 961 files for training.
Found 1372 files belonging to 5 classes.
Using 411 files for validation.
Validation Data: Counter({np.int32(4): 50, np.int32(1): 45, np.int32(3): 42, np.int32(2): 38, np.int32(0): 33}) -> size = 208
Testing Data: Counter({np.int32(4): 51, np.int32(3): 50, np.int32(1): 37, np.int32(2): 33, np.int32(0): 32}) -> size = 203

Training Model_1 | Seed: 1 | Batch: 16 | Epochs: 2
Epoch 1/2
61/61 ━━━━━━━━━━━━━━━━━━━━ 24s 359ms/step - accuracy: 0.1958 - loss: 182.8739 - val_accuracy: 0.1827 - val_loss: 1.6090
Epoch 2/2
61/61 ━━━━━━━━━━━━━━━━━━━━ 23s 373ms/step - accuracy: 0.2506 - loss: 1.6345 - val_accuracy: 0.1827 - val_loss: 1.6068
{'model': 'Model_1', 'seed': 1, 'batch_size': 16, 'epochs': 2, 'val_acc': 0.18269230425357819, 'test_acc': 0.18719211220741272, 'test_loss': 1.6084591150283813}

Training Model_1 | Seed: 1 | Batch: 16 | Epochs: 3
Epoch 1/3
61/61 ━━━━━━━━━━━━━━━━━━━━ 23s 331ms/step - accuracy: 0.2085 - loss: 105.9

In [19]:
results_df = pd.DataFrame(results)
results_df.sort_values(by='test_acc', ascending=False, inplace=True)
results_df.to_csv('grid_search_results.csv', index=False)

print("\nTop Results:")
results_df.head()


Top Results:


,model,seed,batch_size,epochs,val_acc,test_acc,test_loss
5,Model_1,1,32,4,0.265625,0.301370,1.902266
3,Model_1,1,32,2,0.317708,0.296804,1.567134
14,Model_2,1,16,4,0.235577,0.290640,1.656794
7,Model_1,2,16,3,0.211538,0.285714,1.621725
9,Model_1,2,32,2,0.244792,0.273973,1.594431
